statistikoj laŭ silaboj el skribsistemo Dinu Kevako. Silabo en la skribsistemo estas: [K[K]]V[K[K]], do nepre aperas vokalo, kaj povas esti antaŭita kaj sekvita de 0, 1 aŭ 2 konsonantoj.

## Global Variables Reference

**Global variables** (created in one cell, used in other cells):

| Variable | Cell | Type | Description | Example Value |
|----------|------|------|-------------|--------|
| `g_input_file` | **3** | `str` | Path to input YAML file containing word frequency data | `./data/source1_all_words.yaml` |
| `g_input_file_path` | **3** | `pathlib.PosixPath` | Path object for generating output filenames | Derived from `g_input_file` |
| `g_all_words` | **5** | `dict[str, int]` | Word → frequency mapping from input YAML | Keys: Esperanto words; Values: occurrence counts |
| `g_words_with_syllables` | **10** | `list[tuple]` | Syllable-divided words with frequencies | `[(word: str, syllables_array: list, frequency: int), ...]` |

**Local variables** (created and used within single cells) are documented with inline comments above each assignment.

In [1]:
# constants and imports

# Import KevakIlo library for syllable division;
import sys

# ensures workspace root is on the path so local packages (iloj/) are importable
sys.path.insert(0, ".")
from iloj.kevak_ilo import KevakIlo

from pathlib import Path

# g_input_file: str - Path to YAML file containing word frequency data
g_input_file = "./data/source1_all_words.yaml"
# g_input_file = "./data/source2_test.yaml"

# g_input_file_path: pathlib.PosixPath - Path object for generating output filenames (GLOBAL - used in later cells)
g_input_file_path = Path(g_input_file)


In [2]:
# Parse custom YAML format (word: frequency) into dict; avoids yaml library's false-conversion bug
def read_flat_yml(input_file):
    with open(input_file, "r") as f:
        lines = f.readlines()

    all_data = {}
    for line in lines:
        vorto, kvanto = line.strip().split(": ")
        all_data[vorto] = int(kvanto)

    return all_data


In [3]:
# Load input data from YAML file and display basic statistics

# g_all_words: dict[str, int] - Mapping of Esperanto words to their occurrence frequencies
g_all_words = read_flat_yml(g_input_file)
print("Length:", len(g_all_words))
print("First 10 items:", list(g_all_words.items())[:10])

Length: 49463
First 10 items: [('la', 612479), ('de', 308999), ('kaj', 219359), ('en', 146802), ('al', 96398), ('estas', 71030), ('ne', 68672), ('mi', 57988), ('por', 57813), ('li', 56887)]


In [4]:
# Analyze word structure: compute distributions by word length and letters
def analyze_word_structure(word_freq_dict):
    """
    Analyzes the structure of words in the frequency dictionary.

    Args:
        word_freq_dict: Dictionary with word -> frequency mapping

    Returns:
        Tuple of three dicts: (length_distribution, syllable_distribution, letter_distribution)
        - length_distribution: char_count -> total_frequency
        - syllable_distribution: syllable_count -> total_frequency
        - letter_distribution: letter -> total_count across all words
    """
    length_dist = {}
    letter_dist = {}

    for word, freq in word_freq_dict.items():
        clean_word = word.replace("_", "").replace("'", "")
        word_len = len(clean_word)
        length_dist[word_len] = length_dist.get(word_len, 0) + freq

        for char in clean_word:
            letter_dist[char] = letter_dist.get(char, 0) + freq

    return length_dist, letter_dist


In [5]:
# Print word length and letter distributions with totals and percentages
def print_word_syllable_and_letter_distributions():
    length_distribution, letter_distribution = analyze_word_structure(g_all_words)

    total_words = sum(length_distribution.values())
    total_letters = sum(letter_distribution.values())

    print(f"\nTotal words:    {total_words:8d}")
    print(f"Total letters:  {total_letters:8d}")

    print("\n=== WORD LENGTH DISTRIBUTION ===")
    print("Length (chars) -> Total Frequency")
    length_pct_sum = 0
    for length in sorted(length_distribution.keys(), key=lambda x: length_distribution[x], reverse=True):
        count = length_distribution[length]
        pct = count / total_words * 100
        length_pct_sum += pct
        print(f"  {length:2d} chars: {count:8d} : {pct:.3f}%")

    print("\n=== LETTER DISTRIBUTION ===")
    print("Letter -> Total Count")
    letter_pct_sum = 0
    for letter, count in sorted(letter_distribution.items(), key=lambda x: x[1], reverse=True):
        pct = count / total_letters * 100
        letter_pct_sum += pct
        print(f"  {letter}: {count:8d} : {pct:.3f}%")


print_word_syllable_and_letter_distributions()



Total words:     6050218
Total letters:  28844862

=== WORD LENGTH DISTRIBUTION ===
Length (chars) -> Total Frequency
   2 chars:  1691277 : 27.954%
   3 chars:   951134 : 15.721%
   5 chars:   736795 : 12.178%
   6 chars:   538339 : 8.898%
   4 chars:   533533 : 8.818%
   7 chars:   530369 : 8.766%
   8 chars:   434012 : 7.173%
   9 chars:   284081 : 4.695%
  10 chars:   179970 : 2.975%
  11 chars:    91039 : 1.505%
  12 chars:    41541 : 0.687%
  13 chars:    21987 : 0.363%
  14 chars:    10549 : 0.174%
  15 chars:     3905 : 0.065%
  16 chars:     1192 : 0.020%
  17 chars:      365 : 0.006%
  18 chars:      103 : 0.002%
  19 chars:       27 : 0.000%

=== LETTER DISTRIBUTION ===
Letter -> Total Count
  a:  3635399 : 12.603%
  i:  2729894 : 9.464%
  e:  2667569 : 9.248%
  o:  2618592 : 9.078%
  n:  2238590 : 7.761%
  l:  1827293 : 6.335%
  r:  1699426 : 5.892%
  s:  1652446 : 5.729%
  t:  1558867 : 5.404%
  k:  1222870 : 4.239%
  j:  1061388 : 3.680%
  d:   970082 : 3.363%
  u:   886

In [6]:
# Write syllable breakdown of each word to CSV file
def skribu_silabojn_csv(rezultoj: list, output_file: str) -> None:
    """
    Writes the list of triples (word, silaboj, frequency) as CSV into output_file.
    Columns: vorto,silaboj,frekvenco
    """
    ilo = KevakIlo()
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("vorto,silaboj,frekvenco\n")
        for vorto, silaboj, frekvenco in rezultoj:
            formatigita = ilo.formatigi([silaboj])
            f.write(f"{vorto},{formatigita},{frekvenco}\n")
    print(f"Skribis {len(rezultoj)} vortojn en '{output_file}'")


In [7]:
# Parse all words into syllable structures
def analizu_silabojn(all_data: dict) -> list:
    """
    Given all_data dict (word -> frequency, sorted desc by frequency),
    for each word calls KevakIlo.dividuJeSilaboj and builds a list of triples:
    (word, array of silaboj, frequency).
    The list preserves the input order (desc by frequency).
    """
    ilo = KevakIlo()
    rezultoj = []
    for vorto, frekvenco in all_data.items():
        silaboj = ilo.dividuJeSilaboj(vorto)
        # dividuJeSilaboj returns list of Vorto (one per space-separated word);
        # since each entry is a single word, take the first element.
        vorto_silaboj = silaboj[0] if silaboj else []
        rezultoj.append((vorto, vorto_silaboj, frekvenco))
    return rezultoj



In [8]:
# Execute main pipeline: parse words to syllables and export to CSV

# g_words_with_syllables: list[tuple] - List of (word, syllables_array, frequency) tuples with complete syllable breakdown
g_words_with_syllables = analizu_silabojn(g_all_words)

skribu_silabojn_csv(g_words_with_syllables, str(g_input_file_path.with_stem(f"{g_input_file_path.stem}_kevako").with_suffix(".csv")))

Skribis 49463 vortojn en 'data/source1_all_words_kevako.csv'


In [9]:
# Calculate distinct syllables and their frequencies
def kalkulu_distinktajn_silabojn(silaboj: list) -> dict:
    """
    Given a list of triples (vorto, vorto_silaboj, frekvenco),
    returns a dict mapping each distinct syllable string to its total frequency.
    A syllable appears as many times as the word it belongs to was used.
    """
    ilo = KevakIlo()
    result = {}
    for _, vorto_silaboj, frekvenco in silaboj:
        for silabo in vorto_silaboj:
            key = ilo.formatigi([[silabo]])
            result[key] = result.get(key, 0) + frekvenco
    return result

In [10]:
# Write distinct syllables to CSV with cumulative percentage milestones

def write_syllables_to_csv(dist, output_file):
    total_sil_freq = sum(dist.values())
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("silabo,frekvenco,procento\n")
        cumulative = 0
        next_threshold = 10
        for sil, freq in sorted(dist.items(), key=lambda x: x[1], reverse=True):
            pct = freq / total_sil_freq * 100
            f.write(f"{sil},{freq},{pct:.4f}\n")
            cumulative += pct
            while next_threshold <= 90 and cumulative >= next_threshold:
                f.write(f"# --- ĉi tiuj silaboj kune kovras {cumulative:.4f}% (sojlo: {next_threshold}%) ---\n")
                next_threshold += 10
    print(f"Skribis {len(dist)} distinktajn silabojn en '{output_file}'")


# dist: dict[str, int] - Distinct syllables → frequencies; key format: "(k|v|f)" where k=onset, v=vowel, f=finale (local to this cell)
distinktaj_silaboj = kalkulu_distinktajn_silabojn(g_words_with_syllables)

write_syllables_to_csv(distinktaj_silaboj, str(g_input_file_path.with_stem(f"{g_input_file_path.stem}_distinktaj_silaboj").with_suffix(".csv")))

Skribis 1783 distinktajn silabojn en 'data/source1_all_words_distinktaj_silaboj.csv'


In [11]:
# Analyze initial consonant clusters (onsets) of first syllables
def unuaj_komencoj(silaboj: list, multsilabaj: bool = True, unusilabaj: bool = True) -> dict:
    """
    Returns a dict mapping each distinct onset (k-part) of the first syllable to its total frequency.
    - unusilabaj=True: include one-syllable words
    - multsilabaj=True: include words with 2+ syllables
    - both False: nothing
    - both True: include all words (default)
    """
    result = {}
    if not unusilabaj and not multsilabaj:
        return result

    for _, vorto_silaboj, frekvenco in silaboj:
        if not vorto_silaboj:
            continue
        if len(vorto_silaboj) == 1 and not unusilabaj:
            continue
        if len(vorto_silaboj) > 1 and not multsilabaj:
            continue
        komenco = vorto_silaboj[0]['k'] or '∅'
        result[komenco] = result.get(komenco, 0) + frekvenco
    return result


def printu_komencojn(titolo: str, d: dict) -> None:
    total = sum(d.values())
    print(f"\n{titolo}")
    print(f"Distinktaj komencoj: {len(d)}")
    print(f"\n{'Komenco':<12} {'Frekvenco':>10}   {'%':>8}")
    print("-" * 34)
    for komenco, freq in sorted(d.items(), key=lambda x: x[1], reverse=True):
        pct = freq / total * 100
        print(f"  {komenco:<10} {freq:>10}   {pct:>7.3f}%")
        if 'ĥ' in komenco:
            print("  [la ceteraj sube estas pli maloftaj ol 'ĥ']")
            break


# komencoj_1sil: dict[str, int] - Initial consonant clusters in 1-syllable words → frequencies (local to this cell)
printu_komencojn("=== KOMENCOJ DE UNUSILABAJ VORTOJ ===", unuaj_komencoj(g_words_with_syllables, multsilabaj=False))

# komencoj_ĉiuj: dict[str, int] - Initial consonant clusters in all words → frequencies (local to this cell)
printu_komencojn("=== KOMENCOJ DE UNUAJ SILABOJ (ĉiuj vortoj) ===", unuaj_komencoj(g_words_with_syllables))


=== KOMENCOJ DE UNUSILABAJ VORTOJ ===
Distinktaj komencoj: 26

Komenco       Frekvenco          %
----------------------------------
  l              689392    28.784%
  d              362803    15.148%
  ∅              318570    13.301%
  k              308538    12.882%
  n              127481     5.323%
  s               94093     3.929%
  p               92498     3.862%
  m               73893     3.085%
  pr              60223     2.515%
  ĉ               58011     2.422%
  ĝ               41142     1.718%
  pl              41077     1.715%
  v               37036     1.546%
  j               31676     1.323%
  ŝ               20698     0.864%
  tr              20635     0.862%
  kv               4287     0.179%
  t                3754     0.157%
  h                3380     0.141%
  kr               2043     0.085%
  ĵ                1324     0.055%
  f                1274     0.053%
  c                1151     0.048%
  brn                18     0.001%
  r                  16   

In [12]:
# Analyze last syllables in multi-syllable words
def lastaj_silaboj(silaboj: list) -> dict:
    """
    Returns a dict mapping each distinct last syllable (formatted string)
    to its total frequency. Only words with 2+ syllables are considered.
    """
    ilo = KevakIlo()
    result = {}
    for _, vorto_silaboj, frekvenco in silaboj:
        if not vorto_silaboj or len(vorto_silaboj) < 2:
            continue
        key = ilo.formatigi([[vorto_silaboj[-1]]])
        result[key] = result.get(key, 0) + frekvenco
    return result


def printu_silabojn(titolo: str, d: dict, limo: int | None = None) -> None:
    total = sum(d.values())
    print(f"\n{titolo}")
    print(f"Distinktaj silaboj: {len(d)}")
    print(f"\n{'Silabo':<20} {'Frekvenco':>10}   {'%':>8}")
    print("-" * 42)
    for sil, freq in list(sorted(d.items(), key=lambda x: x[1], reverse=True))[:limo]:
        pct = freq / total * 100
        print(f"  {sil:<18} {freq:>10}   {pct:>7.3f}%")
    if limo:
        print(f"  ... (montrante {limo} el {len(d)})")


# lastaj_mult: dict[str, int] - Last syllables in 2+ syllable words → frequencies (local to this cell)
printu_silabojn("=== LASTAJ SILABOJ (nur 2+ silabaj vortoj) ===", lastaj_silaboj(g_words_with_syllables), limo=20)


=== LASTAJ SILABOJ (nur 2+ silabaj vortoj) ===
Distinktaj silaboj: 501

Silabo                Frekvenco          %
------------------------------------------
  (t|o)                  141324     3.866%
  (∅|o)                  126415     3.459%
  (∅|a)                  106038     2.901%
  (t|a|s)                 97417     2.665%
  (t|a)                   94354     2.581%
  (r|o)                   90610     2.479%
  (∅|u)                   77284     2.114%
  (t|o|j)                 73079     1.999%
  (n|o)                   71987     1.969%
  (d|o)                   71840     1.965%
  (t|e)                   70913     1.940%
  (t|i|s)                 67759     1.854%
  (l|o)                   56836     1.555%
  (t|a|j)                 48729     1.333%
  (∅|a|j)                 44015     1.204%
  (m|o)                   42977     1.176%
  (∅|u|j)                 42782     1.170%
  (∅|e|l)                 42673     1.167%
  (r|i|s)                 40044     1.096%
  (∅|a|n)               

In [13]:
# Extract word endings (V, VK, VKK) from last syllables of multi-syllable words
def lastaj_finaĵoj(silaboj: list) -> dict:
    """
    For each word with 2+ syllables, takes the last syllable and extracts
    its ending: V (vowel only), VK (vowel + 1 consonant), or VKK (vowel + 2 consonants).
    Returns a dict: ending_string -> total frequency.
    """
    result = {}
    for _, vorto_silaboj, frekvenco in silaboj:
        if len(vorto_silaboj) < 2:
            continue
        last = vorto_silaboj[-1]
        ending = last['v'] + (last['f'] or '')
        result[ending] = result.get(ending, 0) + frekvenco
    return result


def analizu_lastajn_finaĵojn(silaboj: list) -> None:
    """Analyzes and prints word endings (V, VK, VKK) from last syllables of multi-syllable words."""
    finajoj = lastaj_finaĵoj(silaboj)
    total_finajoj = sum(finajoj.values())
    print(f"Distinktaj lastaj finaĵoj (el 2+ silabaj vortoj): {len(finajoj)}")
    print(f"\n{'Finaĵo':<12} {'Frekvenco':>10}   {'%':>8}")
    print("-" * 34)
    for fin, freq in sorted(finajoj.items(), key=lambda x: x[1], reverse=True):
        pct = freq / total_finajoj * 100
        print(f"  {fin:<10} {freq:>10}   {pct:>7.3f}%")


# Call the analysis function to hide local variables
analizu_lastajn_finaĵojn(g_words_with_syllables)

Distinktaj lastaj finaĵoj (el 2+ silabaj vortoj): 36

Finaĵo        Frekvenco          %
----------------------------------
  o              856979    23.446%
  a              409927    11.215%
  oj             299259     8.187%
  is             292337     7.998%
  as             283175     7.747%
  e              256007     7.004%
  on             210393     5.756%
  i              183828     5.029%
  aj             176814     4.837%
  u              123674     3.384%
  an              93336     2.554%
  ojn             83498     2.284%
  aŭ              66982     1.833%
  ajn             42965     1.175%
  uj              42929     1.174%
  el              42673     1.167%
  os              40848     1.118%
  am              34689     0.949%
  un              18268     0.500%
  us              17908     0.490%
  er              15380     0.421%
  en              14258     0.390%
  ujn             10063     0.275%
  om               9942     0.272%
  in               7516     0.206%
 

In [14]:
# Calculate all consecutive syllable pairs in multi-syllable words
def silabaj_paroj(silaboj: list) -> dict:
    """
    For each word with 2+ syllables, iterates over all consecutive syllable pairs
    and accumulates their frequency.
    Returns dict: (sil1_str, sil2_str) -> total frequency.
    Simple approach first - optimize later if needed.
    """
    ilo = KevakIlo()
    result = {}
    for _, vorto_silaboj, frekvenco in silaboj:
        if len(vorto_silaboj) < 2:
            continue
        for i in range(len(vorto_silaboj) - 1):
            s1 = ilo.skribu([[vorto_silaboj[i]]])
            s2 = ilo.skribu([[vorto_silaboj[i + 1]]])
            key = (s1, s2)
            result[key] = result.get(key, 0) + frekvenco
    return result


def skribu_silabajn_parojn(paroj: dict) -> None:
    """Write consecutive syllable pairs to CSV file with cumulative percentage milestones."""
    output_file_paroj = str(g_input_file_path.with_stem(f"{g_input_file_path.stem}_silabaj_paroj").with_suffix(".csv"))
    total_paroj = sum(paroj.values())
    with open(output_file_paroj, "w", encoding="utf-8") as f:
        f.write("silabo1,silabo2,frekvenco,procento\n")
        cumulative = 0
        next_threshold = 10
        for (s1, s2), freq in sorted(paroj.items(), key=lambda x: x[1], reverse=True):
            pct = freq / total_paroj * 100
            f.write(f"{s1},{s2},{freq},{pct:.4f}\n")
            cumulative += pct
            while next_threshold <= 90 and cumulative >= next_threshold:
                f.write(f"# --- ĉi tiuj paroj kune kovras {cumulative:.4f}% (sojlo: {next_threshold}%) ---\n")
                next_threshold += 10
    print(f"Skribis {len(paroj)} parojn en '{output_file_paroj}'")

skribu_silabajn_parojn(silabaj_paroj(g_words_with_syllables))

Skribis 25147 parojn en 'data/source1_all_words_silabaj_paroj.csv'


In [15]:
# Analyze syllables with 2-char onset AND 2-char finale (5 letters total)
def filter_silabojn_kun_5_literoj(silaboj) -> dict:
    ilo = KevakIlo()
    dist_syllables = {}
    for _, vorto_silaboj, frekvenco in silaboj:
        for silabo in vorto_silaboj:
            # Filter syllables where both k (onset) and f (final consonants) parts have exactly 2 characters
            if len(silabo.get('k') or '') == 2 and len(silabo.get('f') or '') == 2:
                key = ilo.formatigi([[silabo]])
                dist_syllables[key] = dist_syllables.get(key, 0) + frekvenco
    return dist_syllables


def analizu_silabojn_kun_5_literoj(silaboj: list) -> None:
    """Analyzes and prints syllables with 2-char onset AND 2-char finale."""
    # filtered_dist: dict[str, int] - Syllables with 2-char onset AND 2-char finale → frequencies
    filtered_dist = filter_silabojn_kun_5_literoj(silaboj)
    print(f"\nSilaboj kun 5 literoj: {len(filtered_dist)}")
    print(f"\n{'Silabo':<20} {'Frekvenco':>10}   {'%':>8}")
    print("-" * 42)
    # total_filtered: int - Total filtered syllable frequencies: sum(filtered_dist.values())
    total_filtered = sum(filtered_dist.values())
    # sil: str - Current syllable being printed
    # freq: int - Frequency of current syllable
    # pct: float - Percentage of current syllable relative to total
    for sil, freq in sorted(filtered_dist.items(), key=lambda x: x[1], reverse=True):
        pct = freq / total_filtered * 100
        print(f"  {sil:<18} {freq:>10}   {pct:>7.3f}%")


# Call the analysis function to hide local variables
analizu_silabojn_kun_5_literoj(g_words_with_syllables)


Silaboj kun 5 literoj: 30

Silabo                Frekvenco          %
------------------------------------------
  (tr|a|ns)                4427    54.252%
  (br|o|jn)                 587     7.194%
  (gr|a|nd)                 525     6.434%
  (tr|o|jn)                 493     6.042%
  (gv|o|jn)                 415     5.086%
  (gv|a|jn)                 242     2.966%
  (st|o|jn)                 206     2.525%
  (fr|a|nc)                 203     2.488%
  (kr|i|st)                 143     1.752%
  (pl|o|jn)                 131     1.605%
  (kt|o|jn)                 127     1.556%
  (tr|a|jn)                 107     1.311%
  (fr|e|md)                  92     1.127%
  (kt|a|jn)                  80     0.980%
  (br|a|jn)                  77     0.944%
  (gr|o|jn)                  66     0.809%
  (pl|a|jn)                  60     0.735%
  (tr|e|jn)                  34     0.417%
  (dr|o|jn)                  29     0.355%
  (fr|o|nt)                  29     0.355%
  (gl|a|jn)               

In [16]:
# Analyze word distribution by syllable count and compute average lengths
def kalkulu_silabo_distribuon(silaboj: list) -> tuple:
    """Computes syllable count distribution and totals. Returns (silabo_dist, total_silaboj, total_vortoj)."""
    silabo_dist = {}
    total_silaboj = 0
    for _, vorto_silaboj, frekvenco in silaboj:
        n = len(vorto_silaboj)
        silabo_dist[n] = silabo_dist.get(n, 0) + frekvenco
        total_silaboj += n * frekvenco
    total_vortoj = sum(silabo_dist.values())
    return silabo_dist, total_silaboj, total_vortoj


def printu_silabo_distribuon(silaboj: list, all_data: dict) -> None:
    """Prints syllable count distribution table and average word/syllable length statistics."""
    silabo_dist, total_silaboj, total_vortoj = kalkulu_silabo_distribuon(silaboj)

    print(f"Total syllable occurrences: {total_silaboj:8d}")
    print(f"Total word occurrences:     {total_vortoj:8d}")
    print("\n=== WORD DISTRIBUTION BY SYLLABLE COUNT ===")
    print("Syllables -> Word count : % of all words")
    pct_sum = 0
    for n in sorted(silabo_dist.keys()):
        count = silabo_dist[n]
        pct = count / total_vortoj * 100
        pct_sum += pct
        print(f"  {n} silaboj: {count:8d} : {pct:.3f}%")

    avg_silaboj = total_silaboj / total_vortoj
    print(f"\nMeza vorto-longo (en silaboj): {avg_silaboj:.3f}")

    _, letter_distribution = analyze_word_structure(all_data)
    total_letters = sum(letter_distribution.values())
    print(f"Meza vorto-longo (en literoj):  {total_letters / total_vortoj:.3f}")
    print(f"Meza silabo-longo (en literoj): {total_letters / total_silaboj:.3f}")


printu_silabo_distribuon(g_words_with_syllables, g_all_words)



Total syllable occurrences: 12537986
Total word occurrences:      6050218

=== WORD DISTRIBUTION BY SYLLABLE COUNT ===
Syllables -> Word count : % of all words
  1 silaboj:  2395024 : 39.586%
  2 silaboj:  1730488 : 28.602%
  3 silaboj:  1213326 : 20.054%
  4 silaboj:   539230 : 8.913%
  5 silaboj:   150109 : 2.481%
  6 silaboj:    19845 : 0.328%
  7 silaboj:     2095 : 0.035%
  8 silaboj:      101 : 0.002%

Meza vorto-longo (en silaboj): 2.072
Meza vorto-longo (en literoj):  4.768
Meza silabo-longo (en literoj): 2.301


## Notebook Summary

**Purpose:** Comprehensive syllable analysis of Esperanto words using Dinu Kevako's syllable system (format: [K[K]]V[K[K]])

**Process:**

1. **Input data:** Read word frequency mapping from YAML file (`./data/source1_all_words.yaml`) into global variable `g_all_words`

2. **Word structure analysis:**
   - Computed word length and letter distributions
   - Displayed word length distribution and letter frequency statistics to output

3. **Syllable division:**
   - Parsed all words into syllable structures using KevakIlo library
   - Stored parsed results (word, syllables, frequency) in global variable `g_words_with_syllables`
  - Exported syllable breakdown to CSV file (`*_kevako.csv`)

4. **Distinct syllable analysis:**
  - Calculated distinct syllables and their frequencies
  - Stored in global variable `distinktaj_silaboj`
  - Exported to CSV file (`*_distinktaj_silaboj.csv`) with cumulative percentage milestones

5. **Initial syllable analysis:**
  - Analyzed initial consonant clusters (onsets) in 1-syllable words → displayed to output
  - Analyzed initial consonant clusters in all words (first syllable only) → displayed to output

6. **Final syllable analysis:**
  - Analyzed last syllables in multi-syllable words → displayed to output (top 20)
  - Extracted word endings (V, VK, VKK) from last syllables → displayed to output

7. **Syllable pair analysis:**
  - Calculated all consecutive syllable pairs in multi-syllable words
  - Exported to CSV file (`*_silabaj_paroj.csv`) with cumulative percentage milestones

8. **Five-letter syllable analysis:**
  - Analyzed syllables with exactly 2-character onset AND 2-character finale (5 letters total)
  - Displayed results to output

9. **Word distribution analysis:**
  - Calculated word distribution by syllable count
  - Computed average word length (in syllables and letters)
  - Computed average syllable length (in letters)
  - Displayed all statistics to output

**Output files generated:**
- `*_kevako.csv` — Individual word syllable breakdowns
- `*_distinktaj_silaboj.csv` — Distinct syllables with frequencies
- `*_silabaj_paroj.csv` — Consecutive syllable pairs with frequencies
